# Viagens a serviço do governo federal em 2023

Este notebook explora como a quantidade, a duração e as despesas de viagens se distribuem entre cargos públicos. A análise usa dados do Portal da Transparência e apresenta apenas resumos agregados, sem linhas individuais.

**Perguntas:** quais cargos concentram mais viagens? Como variam a quantidade de viagens e a mediana das despesas? Como a mediana das despesas muda entre faixas de duração?

Para executar, coloque `2023_Viagem.csv` em `data/raw/`. Os dados não estão incluídos neste repositório.


## 1. Carregar e preparar os dados

O arquivo original contém informações de identificação e detalhes de itinerário. Primeiro calculamos a duração usando as datas; em seguida, removemos os campos pessoais e os detalhes que não são necessários para as perguntas desta análise.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

arquivo = Path("data/raw/2023_Viagem.csv")
if not arquivo.exists():
    raise FileNotFoundError("Coloque 2023_Viagem.csv na pasta data/raw/. Consulte o README.md.")

# Lê somente os campos usados; dados pessoais e detalhes de viagem ficam fora da análise.
colunas = [
    "Cargo", "Período - Data de início", "Período - Data de fim",
    "Valor diárias", "Valor passagens", "Valor outros gastos"
]
df = pd.read_csv(arquivo, sep=";", encoding="Windows-1252", decimal=",", usecols=colunas)

inicio = pd.to_datetime(df["Período - Data de início"], format="%d/%m/%Y", errors="coerce")
fim = pd.to_datetime(df["Período - Data de fim"], format="%d/%m/%Y", errors="coerce")
df["Dias de viagem"] = (fim - inicio).dt.days
df = df.drop(columns=["Período - Data de início", "Período - Data de fim"])


In [ ]:
componentes_despesa = ["Valor diárias", "Valor passagens", "Valor outros gastos"]
df[componentes_despesa] = df[componentes_despesa].apply(pd.to_numeric, errors="coerce")
df["Despesas consideradas"] = df[componentes_despesa].sum(axis=1, min_count=len(componentes_despesa))
viagens_sem_cargo = int(df["Cargo"].isna().sum())
df["Cargo"] = df["Cargo"].fillna("Não identificado")
mascara_sigilo = df["Cargo"].str.contains("sigilo", case=False, na=False)
viagens_cargo_sigilo = int(mascara_sigilo.sum())
mascara_cargo_divulgado = df["Cargo"].ne("Não identificado") & ~mascara_sigilo
total_viagens_cargo_divulgado = int(mascara_cargo_divulgado.sum())

resumo_qualidade = pd.DataFrame({
    "Indicador": [
        "Viagens no arquivo", "Viagens sem cargo informado",
        "Viagens com cargo protegido por sigilo", "Viagens com cargo divulgado",
        "Viagens sem os três valores de despesa", "Viagens sem duração calculável"
    ],
    "Quantidade": [
        len(df), viagens_sem_cargo, viagens_cargo_sigilo, total_viagens_cargo_divulgado,
        int(df["Despesas consideradas"].isna().sum()),
        int(df["Dias de viagem"].isna().sum())
    ]
})
resumo_qualidade


## 2. Comparar os cargos

Cerca de 39% dos registros não informam o cargo. A tabela de qualidade mostra a quantidade exata. Para que “não informado” não apareça como profissão, a comparação usa somente registros com cargo identificado e divulgado. O gráfico mostra os dez cargos mais frequentes entre os que representam pelo menos 1% desse subconjunto.

A despesa considerada é a soma de diárias, passagens e outros gastos, calculada apenas quando os três valores estão disponíveis. Devoluções não são descontadas. Cargos protegidos por sigilo também ficam fora da comparação.


In [ ]:
por_cargo = (
    df.groupby("Cargo", dropna=False)
      .agg(
          viagens=("Cargo", "size"),
          despesa_mediana=("Despesas consideradas", "median"),
          despesa_total=("Despesas consideradas", "sum"),
          duracao_mediana_dias=("Dias de viagem", "median")
      )
      .reset_index()
)
por_cargo["participacao_pct"] = por_cargo["viagens"] / total_viagens_cargo_divulgado * 100
cargos_frequentes = (
    por_cargo.loc[
        (por_cargo["participacao_pct"] >= 1)
        & ~por_cargo["Cargo"].str.contains("sigilo|não identificado", case=False, na=False)
    ]
    .sort_values("viagens", ascending=False)
    .head(10)
    .reset_index(drop=True)
)
cargos_frequentes.round(2)



In [ ]:
fig, ax = plt.subplots(figsize=(10, max(4, 0.36 * len(cargos_frequentes))))
ax.barh(cargos_frequentes["Cargo"], cargos_frequentes["viagens"], color="#176B87")
ax.invert_yaxis()
ax.set_title("Viagens por cargo divulgado (cargos com pelo menos 1%)")
ax.set_xlabel("Número de viagens")
ax.set_ylabel("")
ax.grid(axis="x", linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()



## 3. Despesas por duração

Agrupamos as viagens em faixas de duração e mostramos a mediana das despesas apenas nas faixas com pelo menos 30 registros e valores completos. A mediana reduz a influência de valores muito altos em comparação com a média.


In [ ]:
limites = [-1, 0, 3, 7, 14, 30, 60, 90, 180, np.inf]
rotulos = ["0", "1–3", "4–7", "8–14", "15–30", "31–60", "61–90", "91–180", ">180"]
df["Faixa de duração (dias)"] = pd.cut(df["Dias de viagem"], bins=limites, labels=rotulos)

por_duracao = (
    df.groupby("Faixa de duração (dias)", observed=False)
      .agg(
          viagens=("Cargo", "size"),
          despesa_mediana=("Despesas consideradas", "median")
      )
      .reset_index()
)
por_duracao = por_duracao.loc[
    (por_duracao["viagens"] >= 30) & por_duracao["despesa_mediana"].notna()
]
por_duracao.round(2)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(por_duracao["Faixa de duração (dias)"].astype(str), por_duracao["despesa_mediana"], color="#D27643")
ax.invert_yaxis()
ax.set_title("Mediana das despesas por faixa de duração")
ax.set_xlabel("Despesa mediana registrada (R$)")
ax.set_ylabel("Duração (dias)")
ax.grid(axis="x", linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()


## Limites da análise

Os dados são registros administrativos e podem conter valores ausentes ou inconsistências. A duração é a diferença entre as datas de início e fim, em dias; os gastos não descontam devoluções. O total por cargo pode incluir apenas as despesas com os três componentes preenchidos; a tabela de qualidade mostra quantos registros ficam sem esses valores completos. Comparações entre grupos descrevem padrões e não demonstram causa e efeito.

**Fonte:** [Portal da Transparência — viagens a serviço](https://portaldatransparencia.gov.br/download-de-dados/viagens). O arquivo de dados deve ser obtido diretamente da fonte.
